# Transformer unlearning baseline on TOFU (Pythia-160M) - v2

Capstone baseline for unlearning in state-space models. This notebook trains (or reuses) a *target* model that memorised
a forget set, trains an *oracle* that never saw it, then compares four unlearning methods against the oracle.

**How to run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. Do not edit any cell except the
two settings in the "Settings" cell. If Colab disconnects, reconnect and **Run all again**: every stage skips what is
already saved in Google Drive and resumes (fine-tuning resumes from its last epoch).

| Stage | What it does | Time on a T4 (estimate) |
|---|---|---|
| 0 | GPU / disk / speed check | 1-2 min |
| 1, 1b | data splits, TOFU perturbed splits | 1-2 min |
| 2, 3 | target + oracle (skipped if already in Drive) | 0 if reused, otherwise 1-2 h |
| 4 | unlearning sweep: 4 methods x 3 learning rates + control | 30-60 min |
| 5 | full evaluation of 8 models | 10-15 min |
| 6 | relearning probe | 2-5 min |
| 7 | report + figures | 1 min |


## Settings

In [ ]:
# RECIPE = "standard": use the standard fine-tune recipe. If a trained target/oracle already exists in
#                      Drive folder ssu_tf (from your earlier run) they are REUSED, so stages 2-3 are skipped.
# RECIPE = "gentle":   retrain target + oracle with a gentler learning rate in a separate folder (ssu_tf_gentle).
#                      Costs ~1-2 extra hours. Use it only after the "standard" run is complete.
RECIPE = "standard"
SEED = 42

## Setup (Drive, packages)

In [ ]:
import os, sys, shutil, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = os.path.expanduser("~/ssu_local")
os.makedirs(BASE, exist_ok=True)
%pip install -q transformers datasets accelerate scikit-learn scipy rouge-score matplotlib pandas tqdm
import torch
assert RECIPE in ("standard", "gentle"), "RECIPE must be 'standard' or 'gentle'"
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")

## The library
All the logic lives in this one cell (written to the Colab machine, then imported). Cells below only call it.

In [ ]:
%%writefile /content/ssu_lib.py
"""ssu_lib.py (v2) - Transformer unlearning baseline on TOFU, with TOFU-style metrics.

Every stage of the study is one function here. The notebook only calls them.
Design rules: fp32 by default, plain PyTorch loops, every stage idempotent
(skips if its output already exists) so a Colab disconnect never costs more than
the stage that was running.
"""
import os, sys, json, time, math, random, gc, shutil, inspect, datetime, subprocess, collections, re
import numpy as np
import torch
import torch.nn.functional as F

# --------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------
DEFAULT_CFG = dict(
    root=None,                      # set by notebook (Drive folder)
    model_name="EleutherAI/pythia-160m",
    seed=42, max_len=128,
    allow_cpu=False,                # smoke test only
    # expected split sizes (TOFU forget05 / retain95 minus a 200-QA holdout)
    n_forget=200, n_holdout=200, n_retain_train=3600, n_probe=200, n_rouge=100,
    # fine-tuning (target + oracle)
    ft_lr=1e-4, ft_bs=16, ft_epochs=10, ft_epochs_retry=15, ft_eval_every=2,
    # gates
    gate_mia=0.80, gate_ratio=0.70, gate_oracle_lo=0.35, gate_oracle_hi=0.65,
    bench_epoch_limit_min=8.0,
    # unlearning (v2: gentler learning rates, finer evaluation, early stop after the operating point)
    tag="v2",                       # sub-folder for everything produced by stages 4-7
    methods=["ga", "graddiff", "npo", "npo_rt"], un_lrs=[3e-6, 1e-5, 3e-5], main_lr=1e-5,
    un_bs=8, un_max_steps=240, un_eval_every=3, un_warmup=5, post_op_steps=9,
    npo_beta=0.1, gd_alpha=1.0, rt_alpha=1.0, diverge_nll=30.0, collapse_rise=8.0, clip=1.0,
    # evaluation
    eval_bs=32, gen_bs=32, gen_max_new=50, mink_frac=0.2,
    # relearning probe
    relearn_n=20, relearn_steps=10, relearn_lr=2e-5, relearn_bs=10, relearn_rouge_n=100,
    # verdict thresholds
    v_mia=0.60, v_retain_rise=0.15, v_rouge_margin=0.05, v_fq=0.05, v_relearn_margin=0.5, v_mu_frac=0.8,
)

LOADER = None      # test hook: callable(name)->list[dict(q,a)]
LOADER_RAW = None  # test hook: callable(name)->list[raw row dicts] (perturbed splits)


class GateFailed(Exception):
    pass


class StopForTest(Exception):
    pass


def make_cfg(**overrides):
    cfg = json.loads(json.dumps(DEFAULT_CFG))
    cfg.update(overrides)
    return cfg


# --------------------------------------------------------------------------------------
# Small utilities
# --------------------------------------------------------------------------------------
def rp(cfg, *parts):
    p = os.path.join(cfg["root"], *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


def log(cfg, msg):
    line = f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(rp(cfg, "RUNLOG.md"), "a") as f:
        f.write(line + "\n")


def _jdefault(o):
    if hasattr(o, "item"):
        return o.item()
    if hasattr(o, "tolist"):
        return o.tolist()
    return str(o)


def jsave(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1, default=_jdefault)
    os.replace(tmp, path)


def jload(path):
    with open(path) as f:
        return json.load(f)


def tp(cfg, kind, *parts):
    """Path inside the tagged output folder (stages 4-7): <root>/<kind>/<tag>/..."""
    return rp(cfg, kind, cfg["tag"], *parts)


def texists(cfg, kind, *parts):
    return os.path.exists(os.path.join(cfg["root"], kind, cfg["tag"], *parts))


def exists(cfg, *parts):
    return os.path.exists(os.path.join(cfg["root"], *parts))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def gate(cfg, name, passed, details):
    path = rp(cfg, "results", "gates.json")
    g = jload(path) if os.path.exists(path) else {}
    g[name] = dict(passed=bool(passed), details=details, time=time.strftime("%Y-%m-%d %H:%M:%S"))
    jsave(path, g)
    log(cfg, f"GATE {name}: {'PASS' if passed else 'FAIL'} | {json.dumps(details, default=_jdefault)}")
    if not passed:
        raise GateFailed(f"{name} failed: {json.dumps(details, default=_jdefault)}")


# --------------------------------------------------------------------------------------
# Model / tokenizer / data encoding
# --------------------------------------------------------------------------------------
def load_tok(name):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_model(path, device):
    from transformers import AutoModelForCausalLM
    try:
        m = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32)
    except (TypeError, ValueError):
        m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32)
    return m.float().to(device)


def encode_rows(tok, rows, max_len):
    """Prompt tokens are masked (-100); loss is on answer tokens (+eos) only (TOFU convention)."""
    out, trunc = [], 0
    for r in rows:
        p = tok(f"Question: {r['q']}\nAnswer:", add_special_tokens=False)["input_ids"]
        a = tok(" " + r["a"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        ids = p + a
        lab = [-100] * len(p) + a
        if len(ids) > max_len:
            ids, lab = ids[:max_len], lab[:max_len]
            trunc += 1
        if len(p) >= len(ids):
            raise ValueError("Prompt alone fills max_len; no answer tokens left. Raise max_len.")
        out.append(dict(ids=ids, labels=lab, q=r["q"], a=r["a"]))
    return out, trunc


def collate(batch, pad_id, device):
    L = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), L), pad_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["ids"])
        ids[i, :n] = torch.tensor(b["ids"])
        lab[i, :n] = torch.tensor(b["labels"])
        att[i, :n] = 1
    return ids.to(device), att.to(device), lab.to(device)


_ATT = {}


def _accepts_mask(model):
    k = type(model).__name__
    if k not in _ATT:
        _ATT[k] = "attention_mask" in inspect.signature(model.forward).parameters
    return _ATT[k]


def token_nll(model, ids, att, lab, amp=False):
    """Per-token NLL [B,T-1] and answer mask [B,T-1]."""
    kw = dict(input_ids=ids)
    if _accepts_mask(model):
        kw["attention_mask"] = att
    if amp:
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**kw).logits
    else:
        logits = model(**kw).logits
    logits = logits[:, :-1].float()
    tgt = lab[:, 1:]
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                          ignore_index=-100, reduction="none").view(tgt.shape)
    return nll, (tgt != -100)


def tok_mean(nll, mask):
    return (nll * mask).sum() / mask.sum()


class Cycler:
    """Endless shuffled stream of items."""

    def __init__(self, items, bs, seed):
        self.items, self.bs, self.rng, self.buf = items, bs, random.Random(seed), []

    def next(self):
        while len(self.buf) < self.bs:
            idx = list(range(len(self.items)))
            self.rng.shuffle(idx)
            self.buf += idx
        take, self.buf = self.buf[: self.bs], self.buf[self.bs:]
        return [self.items[i] for i in take]


# --------------------------------------------------------------------------------------
# Evaluation primitives
# --------------------------------------------------------------------------------------
@torch.no_grad()
def eval_stats(model, ctx, exs):
    """Per-sample answer-token mean NLL and Min-K% logprob, in original order."""
    cfg, device, pad = ctx["cfg"], ctx["device"], ctx["tok"].pad_token_id
    model.eval()
    n = len(exs)
    order = sorted(range(n), key=lambda i: len(exs[i]["ids"]))
    nll_mean, mink = np.zeros(n), np.zeros(n)
    for s in range(0, n, cfg["eval_bs"]):
        idx = order[s: s + cfg["eval_bs"]]
        ids, att, lab = collate([exs[i] for i in idx], pad, device)
        nll, mask = token_nll(model, ids, att, lab)
        nll, mask = nll.cpu().numpy(), mask.cpu().numpy()
        for j, i in enumerate(idx):
            v = nll[j][mask[j]]
            nll_mean[i] = v.mean()
            lp = np.sort(-v)  # ascending logprobs
            k = max(1, int(math.ceil(cfg["mink_frac"] * len(lp))))
            mink[i] = lp[:k].mean()
    return dict(nll=nll_mean, mink=mink)


def auc(member_scores, nonmember_scores):
    from sklearn.metrics import roc_auc_score
    y = np.r_[np.ones(len(member_scores)), np.zeros(len(nonmember_scores))]
    return float(roc_auc_score(y, np.r_[member_scores, nonmember_scores]))


def quick_eval(model, ctx, holdout=None):
    sf = eval_stats(model, ctx, ctx["forget"])
    sh = eval_stats(model, ctx, ctx["holdout"] if holdout is None else holdout)
    sr = eval_stats(model, ctx, ctx["retain_probe"])
    return dict(forget_nll=float(sf["nll"].mean()), holdout_nll=float(sh["nll"].mean()),
                retain_nll=float(sr["nll"].mean()), mia_auc=auc(-sf["nll"], -sh["nll"]))


@torch.no_grad()
def generate_answers(model, ctx, questions):
    cfg, tok, device = ctx["cfg"], ctx["tok"], ctx["device"]
    model.eval()
    prompts = [f"Question: {q}\nAnswer:" for q in questions]
    outs = []
    old = tok.padding_side
    tok.padding_side = "left"
    try:
        for s in range(0, len(prompts), cfg["gen_bs"]):
            chunk = prompts[s: s + cfg["gen_bs"]]
            try:
                enc = tok(chunk, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
                gen = model.generate(**enc, max_new_tokens=cfg["gen_max_new"], do_sample=False,
                                     pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
                outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            except Exception:  # fallback one-by-one (e.g. Mamba + left padding)
                for p in chunk:
                    enc = tok([p], return_tensors="pt", add_special_tokens=False).to(device)
                    gen = model.generate(**enc, max_new_tokens=cfg["gen_max_new"], do_sample=False,
                                         pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
                    outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    finally:
        tok.padding_side = old
    return [o.split("\nQuestion:")[0].strip() for o in outs]


def rouge_recall(preds, golds):
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return float(np.mean([sc.score(g, p)["rougeL"].recall for p, g in zip(preds, golds)]))


# --------------------------------------------------------------------------------------
# Stage 0: environment + benchmark
# --------------------------------------------------------------------------------------
def _bench_one(model, vocab, bs, seq, amp, device, n_warm=2, n_time=5):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)
    hi = max(20, min(5000, vocab - 1))
    ids = torch.randint(10, hi, (bs, seq), device=device)
    att, lab = torch.ones_like(ids), ids.clone()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    times, finite = [], True
    model.train()
    for i in range(n_warm + n_time):
        t = time.time()
        nll, mask = token_nll(model, ids, att, lab, amp)
        loss = tok_mean(nll, mask)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        _sync()
        if i >= n_warm:
            times.append(time.time() - t)
        finite = finite and bool(torch.isfinite(loss).item()) and all(
            bool(torch.isfinite(p.grad).all().item()) for p in model.parameters() if p.grad is not None)
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None
    return dict(s_per_step=float(np.mean(times)), peak_mem_gb=peak, finite=finite)


def stage0(cfg):
    t0 = time.time()
    log(cfg, "STAGE 0: environment + benchmark")
    device = get_device()
    if device.type != "cuda" and not cfg["allow_cpu"]:
        raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU, then re-run.")
    try:
        du = shutil.disk_usage(cfg["root"])
        log(cfg, f"disk free where results are stored: {du.free / 1e9:.1f} GB")
        if du.free < 6e9:
            log(cfg, "WARNING: less than 6 GB free. Stage 4 keeps ~0.65 GB per method. Delete old *_op / *_final "
                     "checkpoints of earlier runs (NOT target_final / oracle_final) to be safe.")
    except Exception:
        pass
    try:
        smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
        print(smi)
    except Exception:
        smi = ""
    import transformers
    info = dict(torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
                device=str(device))
    if device.type == "cuda":
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_mem_gb"] = torch.cuda.get_device_properties(0).total_memory / 1e9
    set_seed(cfg["seed"])
    tok = load_tok(cfg["model_name"])
    model = load_model(cfg["model_name"], device)
    vocab = model.config.vocab_size
    n_train = cfg["n_retain_train"] + cfg["n_forget"]
    res = dict(info=info, model=cfg["model_name"], n_params=sum(p.numel() for p in model.parameters()), runs={})
    bs_chosen = cfg["ft_bs"]
    for bs in sorted({8, bs_chosen}):
        r = _bench_one(model, vocab, bs, 96, False, device)
        r["proj_epoch_min"] = r["s_per_step"] * math.ceil(n_train / bs) / 60
        res["runs"][f"fp32_bs{bs}"] = r
        log(cfg, f"bench fp32 bs{bs}: {r}")
    fp32 = res["runs"][f"fp32_bs{bs_chosen}"]
    use_fp16 = False
    if fp32["proj_epoch_min"] > cfg["bench_epoch_limit_min"] and device.type == "cuda":
        model = load_model(cfg["model_name"], device)
        r = _bench_one(model, vocab, bs_chosen, 96, True, device)
        r["proj_epoch_min"] = r["s_per_step"] * math.ceil(n_train / bs_chosen) / 60
        res["runs"][f"fp16_bs{bs_chosen}"] = r
        log(cfg, f"bench fp16 bs{bs_chosen}: {r}")
        if r["finite"] and r["proj_epoch_min"] < fp32["proj_epoch_min"]:
            use_fp16 = True
    res["use_fp16_ft"] = use_fp16
    res["seconds"] = time.time() - t0
    del model
    free_gpu()
    jsave(rp(cfg, "results", "s0_bench.json"), res)
    best = res["runs"].get(f"fp16_bs{bs_chosen}") if use_fp16 else fp32
    ok = fp32["finite"] and best["proj_epoch_min"] < cfg["bench_epoch_limit_min"] * (1 if device.type == "cuda" else 1e6)
    target_done = os.path.exists(os.path.join(cfg["root"], "ckpt", "target_final", "DONE"))
    if not ok and target_done:
        log(cfg, "G0 would FAIL (slow or non-finite benchmark) but the target model is already trained, so this "
                 "run does not need fine-tuning speed. Continuing.")
        return res
    gate(cfg, "G0", ok, dict(proj_epoch_min=best["proj_epoch_min"], use_fp16_ft=use_fp16,
                              limit_min=cfg["bench_epoch_limit_min"], finite=fp32["finite"]))
    return res


# --------------------------------------------------------------------------------------
# Stage 1: data
# --------------------------------------------------------------------------------------
def load_tofu(name):
    if LOADER is not None:
        return LOADER(name)
    from datasets import load_dataset
    d = load_dataset("locuslab/TOFU", name)
    split = "train" if "train" in d else list(d.keys())[0]
    return [dict(q=r["question"], a=r["answer"]) for r in d[split]]


def _block_purity(rows, start, n_blocks, bs=20):
    ps = []
    for b in range(n_blocks):
        blk = rows[start + b * bs: start + (b + 1) * bs]
        if not blk:
            continue
        cnt = collections.Counter()
        for r in blk:
            cnt.update(set(re.findall(r"\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b", r["q"] + " " + r["a"])))
        ps.append(cnt.most_common(1)[0][1] / len(blk) if cnt else 0.0)
    return float(np.mean(ps)) if ps else 0.0


def stage1(cfg):
    log(cfg, "STAGE 1: data")
    if exists(cfg, "splits", "meta.json") and exists(cfg, "results", "gates.json") and \
            jload(rp(cfg, "results", "gates.json")).get("G1", {}).get("passed"):
        log(cfg, "G1 already passed, skipping")
        return jload(rp(cfg, "splits", "meta.json"))
    forget = load_tofu("forget05")
    retain95 = load_tofu("retain95")
    extra = {}
    for k in ("real_authors", "world_facts"):
        try:
            extra[k] = load_tofu(k)
        except Exception as e:
            log(cfg, f"WARNING: could not load {k} ({e}); its metric will be skipped")
            extra[k] = None
    nh = cfg["n_holdout"]
    purity = _block_purity(retain95, len(retain95) - nh, nh // 20)
    if purity >= 0.5:
        holdout_idx = list(range(len(retain95) - nh, len(retain95)))
        mode = "last_blocks"
    else:
        holdout_idx = random.Random(cfg["seed"]).sample(range(len(retain95)), nh)
        mode = "random_fallback"
        log(cfg, f"WARNING: author-block purity {purity:.2f} < 0.5; using random holdout (authors may be shared)")
    hs = set(holdout_idx)
    holdout = [retain95[i] for i in holdout_idx]
    retain_train = [r for i, r in enumerate(retain95) if i not in hs]
    probe_idx = random.Random(cfg["seed"]).sample(range(len(retain_train)), min(cfg["n_probe"], len(retain_train)))
    retain_probe = [retain_train[i] for i in probe_idx]

    key = lambda r: r["q"] + "\x1f" + r["a"]
    sets = dict(forget=set(map(key, forget)), holdout=set(map(key, holdout)), retain_train=set(map(key, retain_train)))
    disjoint = not (sets["forget"] & sets["holdout"]) and not (sets["forget"] & sets["retain_train"]) \
        and not (sets["holdout"] & sets["retain_train"])

    tok = load_tok(cfg["model_name"])
    max_len = cfg["max_len"]
    for attempt in range(2):
        _, t1 = encode_rows(tok, forget + retain95, max_len)
        rate = t1 / len(forget + retain95)
        if rate < 0.01:
            break
        log(cfg, f"truncation rate {rate:.3%} at max_len {max_len}; raising to 192")
        max_len = 192
    counts = dict(forget=len(forget), holdout=len(holdout), retain_train=len(retain_train),
                  target_train=len(retain_train) + len(forget), oracle_train=len(retain_train))
    exp = dict(forget=cfg["n_forget"], holdout=cfg["n_holdout"], retain_train=cfg["n_retain_train"],
               target_train=cfg["n_retain_train"] + cfg["n_forget"], oracle_train=cfg["n_retain_train"])
    meta = dict(counts=counts, expected=exp, disjoint=disjoint, truncation_rate=rate, max_len=max_len,
                holdout_mode=mode, author_block_purity=purity,
                holdout_first_questions=[holdout[i]["q"] for i in (0, 19, 20) if i < len(holdout)],
                real_authors_n=None if extra["real_authors"] is None else len(extra["real_authors"]),
                world_facts_n=None if extra["world_facts"] is None else len(extra["world_facts"]))
    jsave(rp(cfg, "splits", "splits.json"), dict(forget=forget, holdout=holdout, retain_train=retain_train,
                                                  retain_probe=retain_probe, real_authors=extra["real_authors"],
                                                  world_facts=extra["world_facts"]))
    jsave(rp(cfg, "splits", "meta.json"), meta)
    gate(cfg, "G1", counts == exp and disjoint and rate < 0.01, meta)
    return meta


def adopt_splits(cfg, src_root):
    """Copy the data splits of an earlier run into a fresh root so both runs use identical data."""
    if os.path.exists(os.path.join(cfg["root"], "splits", "meta.json")):
        return
    for f in ("splits.json", "meta.json"):
        shutil.copy(os.path.join(src_root, "splits", f), rp(cfg, "splits", f))
    meta = jload(rp(cfg, "splits", "meta.json"))
    gate(cfg, "G1", meta["counts"] == meta["expected"] and meta["disjoint"], dict(adopted_from=src_root, **meta))


def make_ctx(cfg):
    meta = jload(rp(cfg, "splits", "meta.json"))
    cfg["max_len"] = meta["max_len"]
    sp = jload(rp(cfg, "splits", "splits.json"))
    tok = load_tok(cfg["model_name"])
    enc = lambda rows: None if rows is None else encode_rows(tok, rows, cfg["max_len"])[0]
    ctx = dict(cfg=cfg, tok=tok, device=get_device(), forget=enc(sp["forget"]), holdout=enc(sp["holdout"]),
               retain_train=enc(sp["retain_train"]), retain_probe=enc(sp["retain_probe"]),
               real_authors=enc(sp["real_authors"]), world_facts=enc(sp["world_facts"]))
    ctx["target_train"] = ctx["retain_train"] + ctx["forget"]
    ctx["oracle_train"] = ctx["retain_train"]
    ctx["rouge_forget"] = ctx["forget"][: cfg["n_rouge"]]
    ctx["rouge_retain"] = ctx["retain_probe"][: cfg["n_rouge"]]
    ctx["holdout_traj"] = ctx["holdout"][::2][:100]          # cheaper trajectory evaluation
    ctx["pert"] = jload(rp(cfg, "splits", "perturbed.json")) if exists(cfg, "splits", "perturbed.json") else None
    return ctx


# --------------------------------------------------------------------------------------
# Stages 2-3: fine-tuning (target, oracle)
# --------------------------------------------------------------------------------------
def _lr_at(step, total, warm, base):
    if step < warm:
        return base * (step + 1) / warm
    prog = (step - warm) / max(1, total - warm)
    return base * 0.5 * (1 + math.cos(math.pi * prog))


def finetune(cfg, ctx, tag, train_ex, epochs):
    final = rp(cfg, "ckpt", f"{tag}_final")
    if os.path.exists(os.path.join(final, "DONE")):
        log(cfg, f"[{tag}] final checkpoint exists, skipping training")
        return final
    device, tok = ctx["device"], ctx["tok"]
    bench = rp(cfg, "results", "s0_bench.json")
    amp = bool(jload(bench).get("use_fp16_ft")) if os.path.exists(bench) and device.type == "cuda" else False
    set_seed(cfg["seed"])
    model = load_model(cfg["model_name"], device)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["ft_lr"], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    N, bs = len(train_ex), cfg["ft_bs"]
    spe = math.ceil(N / bs)
    total, warm = epochs * spe, max(1, int(0.1 * epochs * spe))
    latest = rp(cfg, "ckpt", f"{tag}_latest.pt")
    start, progress = 0, []
    if os.path.exists(latest):
        sd = torch.load(latest, map_location=device, weights_only=False)
        if sd.get("epochs") == epochs:
            model.load_state_dict(sd["model"])
            opt.load_state_dict(sd["opt"])
            start, progress = sd["epoch"], sd["progress"]
            log(cfg, f"[{tag}] resumed from epoch {start}")
        else:
            os.remove(latest)
        del sd
    log(cfg, f"[{tag}] fine-tuning {N} examples, {epochs} epochs, bs {bs}, lr {cfg['ft_lr']}, amp={amp}")
    for ep in range(start, epochs):
        t0 = time.time()
        perm = torch.randperm(N, generator=torch.Generator().manual_seed(cfg["seed"] * 1000 + ep)).tolist()
        run = 0.0
        model.train()
        for bi in range(spe):
            batch = [train_ex[i] for i in perm[bi * bs:(bi + 1) * bs]]
            ids, att, lab = collate(batch, tok.pad_token_id, device)
            for g in opt.param_groups:
                g["lr"] = _lr_at(ep * spe + bi, total, warm, cfg["ft_lr"])
            nll, mask = token_nll(model, ids, att, lab, amp)
            loss = tok_mean(nll, mask)
            if not torch.isfinite(loss):
                raise RuntimeError(f"[{tag}] non-finite loss at epoch {ep + 1} batch {bi}")
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
            scaler.step(opt)
            scaler.update()
            run += loss.item()
        row = dict(epoch=ep + 1, train_loss=run / spe, epoch_seconds=time.time() - t0)
        if (ep + 1) % cfg["ft_eval_every"] == 0 or ep + 1 == epochs:
            row.update(quick_eval(model, ctx))
        progress.append(row)
        log(cfg, f"[{tag}] {row}")
        jsave(rp(cfg, "results", f"progress_{tag}.json"), progress)
        tmp = latest + ".tmp"
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), epoch=ep + 1, progress=progress,
                        epochs=epochs), tmp)
        os.replace(tmp, latest)
        if cfg.get("_test_stop_epoch") == ep + 1:
            raise StopForTest(f"test stop after epoch {ep + 1}")
    model.save_pretrained(final)
    tok.save_pretrained(final)
    open(os.path.join(final, "DONE"), "w").write("ok")
    if os.path.exists(latest):
        os.remove(latest)
    del model, opt
    free_gpu()
    return final


def stage2(cfg):
    log(cfg, "STAGE 2: target model")
    summ_path = rp(cfg, "results", "s2_target_summary.json")
    if os.path.exists(summ_path) and jload(summ_path).get("passed"):
        log(cfg, "G2 already passed, skipping")
        return jload(summ_path)
    ctx = make_ctx(cfg)
    attempts = []
    for attempt, epochs in enumerate([cfg["ft_epochs"], cfg["ft_epochs_retry"]]):
        final = finetune(cfg, ctx, "target", ctx["target_train"], epochs)
        model = load_model(final, ctx["device"])
        sf = eval_stats(model, ctx, ctx["forget"])["nll"]
        sh = eval_stats(model, ctx, ctx["holdout"])["nll"]
        del model
        free_gpu()
        det = dict(epochs=epochs, mia_auc=auc(-sf, -sh), forget_nll=float(sf.mean()), holdout_nll=float(sh.mean()))
        det["ratio"] = det["forget_nll"] / det["holdout_nll"]
        ok = det["mia_auc"] >= cfg["gate_mia"] and det["ratio"] <= cfg["gate_ratio"]
        attempts.append(dict(det, passed=ok))
        log(cfg, f"G2 attempt {attempt + 1}: {det} -> {'PASS' if ok else 'FAIL'}")
        if ok:
            jsave(summ_path, dict(passed=True, epochs_used=epochs, attempts=attempts, target_dir=final))
            gate(cfg, "G2", True, det)
            return jload(summ_path)
        if attempt == 0:
            for suffix in ("final",):
                src = rp(cfg, "ckpt", f"target_{suffix}")
                if os.path.exists(src):
                    shutil.move(src, rp(cfg, "ckpt", f"target_attempt1_failed_{suffix}"))
            pj = rp(cfg, "results", "progress_target.json")
            if os.path.exists(pj):
                shutil.move(pj, rp(cfg, "results", "progress_target_attempt1.json"))
    jsave(summ_path, dict(passed=False, attempts=attempts))
    gate(cfg, "G2", False, attempts[-1])


def stage3(cfg):
    log(cfg, "STAGE 3: oracle")
    summ_path = rp(cfg, "results", "s3_oracle_summary.json")
    if os.path.exists(summ_path) and jload(summ_path).get("passed"):
        log(cfg, "G3 already passed, skipping")
        return jload(summ_path)
    epochs = jload(rp(cfg, "results", "s2_target_summary.json"))["epochs_used"]
    ctx = make_ctx(cfg)
    final = finetune(cfg, ctx, "oracle", ctx["oracle_train"], epochs)
    model = load_model(final, ctx["device"])
    sf = eval_stats(model, ctx, ctx["forget"])["nll"]
    sh = eval_stats(model, ctx, ctx["holdout"])["nll"]
    sr = eval_stats(model, ctx, ctx["retain_probe"])["nll"]
    del model
    free_gpu()
    det = dict(mia_auc=auc(-sf, -sh), forget_nll=float(sf.mean()), holdout_nll=float(sh.mean()),
               retain_nll=float(sr.mean()), epochs=epochs)
    # context only (not gated): AUC of the never-fine-tuned model shows how much the two author groups differ
    # in difficulty by themselves; the oracle should sit near that value, not near the target's (~0.9).
    pm = load_model(cfg["model_name"], ctx["device"])
    pf = eval_stats(pm, ctx, ctx["forget"])["nll"]
    ph = eval_stats(pm, ctx, ctx["holdout"])["nll"]
    det["pretrained_mia_auc"] = auc(-pf, -ph)
    del pm
    free_gpu()
    jsave(rp(cfg, "results", "oracle_forget_nll.json"), sf.tolist())
    ok = cfg["gate_oracle_lo"] <= det["mia_auc"] <= cfg["gate_oracle_hi"]
    jsave(summ_path, dict(passed=ok, **det))
    gate(cfg, "G3", ok, dict(det, band=[cfg["gate_oracle_lo"], cfg["gate_oracle_hi"]]))
    return jload(summ_path)


# --------------------------------------------------------------------------------------
# Stage 1b: TOFU "perturbed" splits (paraphrased + perturbed answers) for truth-ratio metrics
# --------------------------------------------------------------------------------------
PERT_SETS = dict(forget="forget05_perturbed", retain="retain_perturbed",
                 real_authors="real_authors_perturbed", world_facts="world_facts_perturbed")


def load_raw(name):
    if LOADER_RAW is not None:
        return LOADER_RAW(name)
    from datasets import load_dataset
    d = load_dataset("locuslab/TOFU", name)
    split = "train" if "train" in d else list(d.keys())[0]
    return [dict(r) for r in d[split]]


def _parse_pert(r):
    para = r.get("paraphrased_answer")
    pert = r.get("perturbed_answer")
    if pert is None:
        pert = [v for k, v in r.items() if str(k).startswith("perturbed_answer") and isinstance(v, str)]
    if isinstance(pert, str):
        pert = [pert]
    pert = [p for p in (pert or []) if isinstance(p, str) and p.strip()]
    if not isinstance(para, str) or not para.strip() or not pert or "question" not in r or "answer" not in r:
        return None
    return dict(q=r["question"], a=r["answer"], para=para, pert=pert)


def stage1b(cfg):
    """Optional (never blocks the study): without these splits Forget Quality / Model Utility are skipped."""
    log(cfg, "STAGE 1b: perturbed splits for TOFU-style metrics")
    if exists(cfg, "splits", "perturbed.json"):
        log(cfg, "perturbed.json exists, skipping")
        return jload(rp(cfg, "splits", "perturbed_meta.json"))
    sp = jload(rp(cfg, "splits", "splits.json"))
    forget_q = {r["q"] for r in sp["forget"]}
    retain_q = {r["q"] for r in sp["retain_train"]}
    out, meta = {}, {}
    for key, name in PERT_SETS.items():
        try:
            raw = load_raw(name)
        except Exception as e:
            log(cfg, f"WARNING: could not load '{name}' ({type(e).__name__}: {str(e)[:150]}). Metrics using it are skipped.")
            meta[key] = dict(loaded=False, error=str(e)[:200])
            continue
        cols = list(raw[0].keys()) if raw else []
        parsed = [p for p in (_parse_pert(r) for r in raw) if p is not None]
        n_raw = len(raw)
        if key == "forget":
            parsed = [p for p in parsed if p["q"] in forget_q]
        elif key == "retain":
            parsed = [p for p in parsed if p["q"] in retain_q]       # only questions the models were trained on
            if len(parsed) > cfg["n_probe"]:
                idx = random.Random(cfg["seed"]).sample(range(len(parsed)), cfg["n_probe"])
                parsed = [parsed[i] for i in sorted(idx)]
        meta[key] = dict(loaded=True, split=name, columns=cols, n_raw=n_raw, n_used=len(parsed),
                         n_perturbed_per_q=float(np.mean([len(p["pert"]) for p in parsed])) if parsed else 0)
        log(cfg, f"  {name}: columns={cols} raw={n_raw} used={len(parsed)}")
        if parsed:
            out[key] = parsed
    meta["usable_for_model_utility"] = all(k in out for k in ("retain", "real_authors", "world_facts"))
    meta["usable_for_forget_quality"] = "forget" in out
    jsave(rp(cfg, "splits", "perturbed.json"), out)
    jsave(rp(cfg, "splits", "perturbed_meta.json"), meta)
    log(cfg, f"  Model Utility possible: {meta['usable_for_model_utility']} | Forget Quality possible: "
             f"{meta['usable_for_forget_quality']}")
    return meta


# --------------------------------------------------------------------------------------
# TOFU-style metrics + full evaluation of one model
# --------------------------------------------------------------------------------------
def hmean(xs):
    xs = [max(float(x), 1e-12) for x in xs]
    return len(xs) / sum(1.0 / x for x in xs)


def tofu_set_metrics(model, ctx, items, kind):
    """Normalised answer probability, ROUGE-L recall and truth-ratio score for one evaluation set.
    Truth ratio R = mean_k P(perturbed_k | q) / P(paraphrased | q), P = exp(-mean token NLL) (TOFU paper).
    forget set: score = mean(min(R, 1/R)); keep sets: score = mean(max(0, 1-R))."""
    cfg, tok = ctx["cfg"], ctx["tok"]
    ml = cfg["max_len"]
    gt = encode_rows(tok, [dict(q=i["q"], a=i["a"]) for i in items], ml)[0]
    pa = encode_rows(tok, [dict(q=i["q"], a=i["para"]) for i in items], ml)[0]
    flat, owner = [], []
    for k, i in enumerate(items):
        for p in i["pert"]:
            flat.append(dict(q=i["q"], a=p))
            owner.append(k)
    pe = encode_rows(tok, flat, ml)[0]
    p_gt = np.exp(-eval_stats(model, ctx, gt)["nll"])
    p_pa = np.exp(-eval_stats(model, ctx, pa)["nll"])
    p_pe = np.exp(-eval_stats(model, ctx, pe)["nll"])
    mean_pert, cnt = np.zeros(len(items)), np.zeros(len(items))
    for o, p in zip(owner, p_pe):
        mean_pert[o] += p
        cnt[o] += 1
    R = (mean_pert / np.maximum(cnt, 1)) / np.maximum(p_pa, 1e-30)
    if kind == "forget":
        score = np.minimum(R, 1.0 / np.maximum(R, 1e-30))
    else:
        score = np.maximum(0.0, 1.0 - R)
    sub = gt[: cfg["n_rouge"]]
    rouge = rouge_recall(generate_answers(model, ctx, [e["q"] for e in sub]), [e["a"] for e in sub])
    return dict(n=len(items), prob=float(p_gt.mean()), rouge=rouge, truth_ratio_score=float(score.mean()),
                R=R.tolist())


def full_eval(model, ctx, name, oracle_R=None, oracle_forget_nll=None):
    cfg = ctx["cfg"]
    sf = eval_stats(model, ctx, ctx["forget"])
    sh = eval_stats(model, ctx, ctx["holdout"])
    sr = eval_stats(model, ctx, ctx["retain_probe"])
    out = dict(name=name, forget_nll=float(sf["nll"].mean()), holdout_nll=float(sh["nll"].mean()),
               retain_nll=float(sr["nll"].mean()),
               mia_auc=auc(-sf["nll"], -sh["nll"]), mink_auc=auc(sf["mink"], sh["mink"]))
    rf, rr = ctx["rouge_forget"], ctx["rouge_retain"]
    out["forget_rouge"] = rouge_recall(generate_answers(model, ctx, [e["q"] for e in rf]), [e["a"] for e in rf])
    out["retain_rouge"] = rouge_recall(generate_answers(model, ctx, [e["q"] for e in rr]), [e["a"] for e in rr])
    if oracle_forget_nll is not None and name != "oracle":
        from scipy.stats import ks_2samp
        ks = ks_2samp(sf["nll"], np.array(oracle_forget_nll))
        out["ks_stat_nll"], out["ks_p_nll"] = float(ks.statistic), float(ks.pvalue)
    out["forget_nll_per_sample"] = sf["nll"].tolist()
    # TOFU-style metrics (only if the perturbed splits were available)
    out["tofu"], out["model_utility"], out["forget_quality"] = {}, None, None
    pert = ctx.get("pert")
    if pert:
        for key, kind in (("forget", "forget"), ("retain", "keep"), ("real_authors", "keep"), ("world_facts", "keep")):
            if key in pert:
                try:
                    out["tofu"][key] = tofu_set_metrics(model, ctx, pert[key], kind)
                except Exception as e:
                    log(cfg, f"WARNING: TOFU metrics for '{key}' failed ({type(e).__name__}: {str(e)[:120]})")
        t = out["tofu"]
        if all(k in t for k in ("retain", "real_authors", "world_facts")):
            nine = []
            for k in ("retain", "real_authors", "world_facts"):
                nine += [t[k]["prob"], t[k]["rouge"], t[k]["truth_ratio_score"]]
            out["model_utility"] = hmean(nine)                 # TOFU definition; collapses to ~0 if any term is ~0
            out["model_utility_arith"] = float(np.mean(nine))  # robust version used for the verdict
            out["retain_utility"] = float(np.mean([t["retain"]["prob"], t["retain"]["rouge"],
                                                    t["retain"]["truth_ratio_score"]]))
        if "forget" in t and oracle_R is not None and name != "oracle":
            from scipy.stats import ks_2samp
            out["forget_quality"] = float(ks_2samp(np.array(t["forget"]["R"]), np.array(oracle_R)).pvalue)
    return out


# --------------------------------------------------------------------------------------
# Stage 4: unlearning sweep (keeps one checkpoint per method: the best operating point)
# --------------------------------------------------------------------------------------
def run_name(method, lr):
    return f"{method}_lr{lr:g}"


def _promote(cfg, method, name, lr, score, cand):
    """Keep `cand` as the method's best checkpoint if its score (tuple, lower = better) beats the current best."""
    best_path = tp(cfg, "results", f"best_{method}.json")
    cur = jload(best_path) if os.path.exists(best_path) else None
    if cur is not None and tuple(score) >= tuple(cur["score"]) and os.path.isdir(cur["ckpt"]):
        shutil.rmtree(cand, ignore_errors=True)
        return False
    best_dir = tp(cfg, "ckpt", f"best_{method}")
    tmp = best_dir + "_new"
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.move(cand, tmp)
    shutil.rmtree(best_dir, ignore_errors=True)
    os.rename(tmp, best_dir)
    jsave(best_path, dict(method=method, run=name, lr=lr, score=list(score), ckpt=best_dir))
    return True


def run_unlearn(cfg, ctx, method, lr, oracle_mean):
    device, tok = ctx["device"], ctx["tok"]
    name = run_name(method, lr)
    out_json = tp(cfg, "results", f"s4_{name}.json")
    if os.path.exists(out_json):
        log(cfg, f"[{name}] already done, skipping")
        return jload(out_json)
    cand = tp(cfg, "ckpt", f"cand_{name}")
    shutil.rmtree(cand, ignore_errors=True)
    target_dir = jload(rp(cfg, "results", "s2_target_summary.json"))["target_dir"]
    if not os.path.isdir(target_dir):
        target_dir = rp(cfg, "ckpt", "target_final")
    set_seed(cfg["seed"])
    model = load_model(target_dir, device)
    ref = None
    if method == "npo" or method == "npo_rt":
        ref = load_model(target_dir, device).eval()
        for p in ref.parameters():
            p.requires_grad_(False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    fstream = Cycler(ctx["forget"], cfg["un_bs"], cfg["seed"])
    rstream = Cycler(ctx["retain_train"], cfg["un_bs"], cfg["seed"] + 1)
    pad = tok.pad_token_id
    traj, op_step, stop_at, t0 = [], None, None, time.time()
    diverged = collapsed = False

    def do_eval(step, batch_loss):
        model.eval()
        row = dict(step=step, batch_loss=batch_loss, **quick_eval(model, ctx, ctx["holdout_traj"]))
        model.train()
        traj.append(row)
        return row

    base = do_eval(0, None)
    log(cfg, f"[{name}] start: forget {base['forget_nll']:.3f} retain {base['retain_nll']:.3f} "
             f"(oracle forget target {oracle_mean:.3f})")
    model.train()
    for step in range(1, cfg["un_max_steps"] + 1):
        for g in opt.param_groups:
            g["lr"] = lr * min(1.0, step / cfg["un_warmup"])
        fb = collate(fstream.next(), pad, device)
        bn = None
        if method == "ga":
            nll, mask = token_nll(model, *fb)
            bn = tok_mean(nll, mask)
            loss = -bn
        elif method == "graddiff":
            nll, mask = token_nll(model, *fb)
            bn = tok_mean(nll, mask)
            rn, rm = token_nll(model, *collate(rstream.next(), pad, device))
            loss = -bn + cfg["gd_alpha"] * tok_mean(rn, rm)
        elif method in ("npo", "npo_rt"):
            nll, mask = token_nll(model, *fb)
            bn = tok_mean(nll, mask)
            logp_theta = -(nll * mask).sum(1)
            with torch.no_grad():
                rn, rm = token_nll(ref, *fb)
                logp_ref = -(rn * rm).sum(1)
            beta = cfg["npo_beta"]
            loss = -(2.0 / beta) * F.logsigmoid(-beta * (logp_theta - logp_ref)).mean()
            if method == "npo_rt":
                rn2, rm2 = token_nll(model, *collate(rstream.next(), pad, device))
                loss = loss + cfg["rt_alpha"] * tok_mean(rn2, rm2)
        elif method == "retainft":
            rn, rm = token_nll(model, *collate(rstream.next(), pad, device))
            loss = tok_mean(rn, rm)
        else:
            raise ValueError(method)
        lv = float(loss.item())
        bnv = None if bn is None else float(bn.item())
        if not math.isfinite(lv) or (bnv is not None and bnv > cfg["diverge_nll"]):
            diverged = True
            log(cfg, f"[{name}] DIVERGED at step {step} (loss {lv}, forget batch NLL {bnv})")
            break
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
        opt.step()
        if step % cfg["un_eval_every"] == 0:
            row = do_eval(step, lv)
            rise = row["retain_nll"] - base["retain_nll"]
            log(cfg, f"[{name}] step {step}: forget {row['forget_nll']:.3f} retain +{rise:.3f} "
                     f"mia {row['mia_auc']:.3f}")
            if not math.isfinite(row["forget_nll"]) or row["forget_nll"] > cfg["diverge_nll"]:
                diverged = True
                log(cfg, f"[{name}] DIVERGED (eval forget NLL {row['forget_nll']})")
                break
            if op_step is None and row["forget_nll"] >= oracle_mean:
                op_step, stop_at = step, step + cfg["post_op_steps"]
                model.save_pretrained(cand)
                open(os.path.join(cand, "DONE"), "w").write("ok")
                log(cfg, f"[{name}] OPERATING POINT at step {step} (forget NLL reached the oracle's)")
            if op_step is None and rise > cfg["collapse_rise"]:
                collapsed = True
                log(cfg, f"[{name}] COLLAPSED (retain NLL +{rise:.1f}) before reaching the oracle's forget NLL")
                break
            if stop_at is not None and step >= stop_at:
                break
    final_forget = traj[-1]["forget_nll"]
    if op_step is not None:
        op_row = next(r for r in traj if r["step"] == op_step)
        score = (0, op_row["retain_nll"] - base["retain_nll"])
        status = "reached"
    else:
        status = "diverged" if diverged else ("collapsed" if collapsed else "never_reached")
        score = (1, -final_forget)
        if not diverged and not collapsed:
            model.save_pretrained(cand)
            open(os.path.join(cand, "DONE"), "w").write("ok")
    promoted = False
    if os.path.isdir(cand):
        promoted = _promote(cfg, method, name, lr, score, cand)
    res = dict(method=method, lr=lr, status=status, op_step=op_step, diverged=diverged, collapsed=collapsed,
               score=list(score), promoted_to_best=promoted, trajectory=traj, oracle_forget_nll=oracle_mean,
               seconds=time.time() - t0, seed=cfg["seed"], bs=cfg["un_bs"], max_steps=cfg["un_max_steps"])
    jsave(out_json, res)
    del model, ref, opt
    free_gpu()
    log(cfg, f"[{name}] finished: {status}, op_step={op_step}, promoted={promoted}, {res['seconds']:.0f}s")
    return res


def stage4(cfg):
    log(cfg, "STAGE 4: unlearning sweep")
    ctx = make_ctx(cfg)
    oracle_mean = jload(rp(cfg, "results", "s3_oracle_summary.json"))["forget_nll"]
    runs = [(m, lr) for m in cfg["methods"] for lr in cfg["un_lrs"]] + [("retainft", cfg["main_lr"])]
    out = []
    for m, lr in runs:
        out.append(run_unlearn(cfg, ctx, m, lr, oracle_mean))
        if cfg.get("_test_stop_after") == run_name(m, lr):
            raise StopForTest(f"test stop after {run_name(m, lr)}")
    return out


def final_models(cfg):
    """[(method, best_json)] for every method that has a kept checkpoint."""
    out = []
    for m in list(cfg["methods"]) + ["retainft"]:
        p = tp(cfg, "results", f"best_{m}.json")
        if os.path.exists(p):
            b = jload(p)
            if os.path.isdir(b["ckpt"]):
                out.append((m, b))
            else:
                log(cfg, f"WARNING: best checkpoint for {m} is missing on disk ({b['ckpt']}). "
                         f"Delete results/{cfg['tag']}/best_{m}.json and the s4_{m}_*.json files and re-run stage 4.")
        else:
            log(cfg, f"note: no checkpoint kept for {m} (every run diverged or collapsed before the operating point)")
    return out


# --------------------------------------------------------------------------------------
# Stage 5: full evaluation battery
# --------------------------------------------------------------------------------------
def stage5(cfg):
    log(cfg, "STAGE 5: full evaluation battery")
    ctx = make_ctx(cfg)
    dev = ctx["device"]
    target_dir = jload(rp(cfg, "results", "s2_target_summary.json"))["target_dir"]
    if not os.path.isdir(target_dir):
        target_dir = rp(cfg, "ckpt", "target_final")
    oracle_forget_nll = jload(rp(cfg, "results", "oracle_forget_nll.json"))
    targets = [("oracle", rp(cfg, "ckpt", "oracle_final")), ("target", target_dir), ("pretrained", cfg["model_name"])]
    for m, b in final_models(cfg):
        targets.append((m, b["ckpt"]))
    oracle_R = None
    for name, path in targets:
        out = tp(cfg, "results", f"s5_eval_{name}.json")
        if os.path.exists(out):
            log(cfg, f"[eval {name}] exists, skipping")
        else:
            t0 = time.time()
            model = load_model(path, dev)
            res = full_eval(model, ctx, name, oracle_R, oracle_forget_nll)
            res["ckpt"], res["seconds"] = path, time.time() - t0
            jsave(out, res)
            small = {k: v for k, v in res.items() if k not in ("forget_nll_per_sample", "tofu")}
            log(cfg, f"[eval {name}] " + json.dumps(small, default=_jdefault))
            del model
            free_gpu()
        if name == "oracle":
            e = jload(out)
            if e.get("tofu", {}).get("forget"):
                oracle_R = e["tofu"]["forget"]["R"]


# --------------------------------------------------------------------------------------
# Stage 6: relearning probe
# --------------------------------------------------------------------------------------
def relearn_probe(model, ctx):
    cfg, tok, device = ctx["cfg"], ctx["tok"], ctx["device"]
    forget = ctx["forget"]
    idx = random.Random(cfg["seed"]).sample(range(len(forget)), cfg["relearn_n"])
    s = set(idx)
    relearn_set = [forget[i] for i in idx]
    rest = [e for i, e in enumerate(forget) if i not in s]
    pre = float(eval_stats(model, ctx, rest)["nll"].mean())
    set_seed(cfg["seed"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["relearn_lr"], weight_decay=0.0)
    stream = Cycler(relearn_set, cfg["relearn_bs"], cfg["seed"])
    model.train()
    for _ in range(cfg["relearn_steps"]):
        ids, att, lab = collate(stream.next(), tok.pad_token_id, device)
        nll, mask = token_nll(model, ids, att, lab)
        loss = tok_mean(nll, mask)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
        opt.step()
    post = float(eval_stats(model, ctx, rest)["nll"].mean())
    sub = rest[: cfg["relearn_rouge_n"]]
    rouge = rouge_recall(generate_answers(model, ctx, [e["q"] for e in sub]), [e["a"] for e in sub])
    return dict(pre_nll=pre, post_nll=post, post_rouge=rouge, n_relearn=len(relearn_set), n_heldout=len(rest))


def stage6(cfg):
    log(cfg, "STAGE 6: relearning probe")
    ctx = make_ctx(cfg)
    out_path = tp(cfg, "results", "s6_relearn.json")
    res = jload(out_path) if os.path.exists(out_path) else {}
    targets = [("oracle", rp(cfg, "ckpt", "oracle_final"))] + [(m, b["ckpt"]) for m, b in final_models(cfg)]
    for name, path in targets:
        if name in res:
            log(cfg, f"[relearn {name}] exists, skipping")
            continue
        model = load_model(path, ctx["device"])
        res[name] = relearn_probe(model, ctx)
        jsave(out_path, res)
        log(cfg, f"[relearn {name}] {res[name]}")
        del model
        free_gpu()
    return res


# --------------------------------------------------------------------------------------
# Stage 7: report (every number is read from JSON files written by the stages above)
# --------------------------------------------------------------------------------------
METHOD_COLOR = {"ga": "#2a78d6", "graddiff": "#eb6834", "npo": "#1baf7a", "npo_rt": "#e87ba4", "retainft": "#898781"}
METHOD_LABEL = {"ga": "GA", "graddiff": "GradDiff", "npo": "NPO", "npo_rt": "NPO+RT", "retainft": "RetainFT (control)"}
METHOD_MARK = {"ga": "o", "graddiff": "s", "npo": "^", "npo_rt": "D", "retainft": "x"}
LR_STYLE = ["-", "--", ":"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"


def _f3(x, d=3):
    if x is None:
        return "-"
    if isinstance(x, (int, float)):
        if x != 0 and abs(x) < 10 ** (-d):
            return f"{x:.1e}"
        return f"{x:.{d}f}"
    return str(x)


def _md(headers, rows):
    s = "| " + " | ".join(headers) + " |\n|" + "|".join("---" for _ in headers) + "|\n"
    for r in rows:
        s += "| " + " | ".join(str(c) for c in r) + " |\n"
    return s


def _style_ax(ax):
    ax.set_facecolor(SURFACE)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_color("#c3c2b7")
    ax.tick_params(colors=INK2, labelsize=9)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(INK2)
    ax.yaxis.label.set_color(INK2)
    ax.title.set_color(INK)


def stage7(cfg):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    log(cfg, "STAGE 7: report")
    RR = lambda *p: tp(cfg, "results", *p)
    oracle = jload(rp(cfg, "results", "s3_oracle_summary.json"))
    gates = jload(rp(cfg, "results", "gates.json")) if os.path.exists(rp(cfg, "results", "gates.json")) else {}
    bench = jload(rp(cfg, "results", "s0_bench.json")) if os.path.exists(rp(cfg, "results", "s0_bench.json")) else {}
    pmeta = jload(rp(cfg, "splits", "perturbed_meta.json")) if exists(cfg, "splits", "perturbed_meta.json") else {}
    methods = list(cfg["methods"]) + ["retainft"]
    runs = []
    for m in methods:
        for lr in (cfg["un_lrs"] if m != "retainft" else [cfg["main_lr"]]):
            p = RR(f"s4_{run_name(m, lr)}.json")
            if os.path.exists(p):
                runs.append(jload(p))
    best = {m: jload(RR(f"best_{m}.json")) for m in methods if os.path.exists(RR(f"best_{m}.json"))}
    evals = {}
    for nm in ["pretrained", "target", "oracle"] + methods:
        if os.path.exists(RR(f"s5_eval_{nm}.json")):
            evals[nm] = jload(RR(f"s5_eval_{nm}.json"))
    relearn = jload(RR("s6_relearn.json")) if os.path.exists(RR("s6_relearn.json")) else {}
    fig_path = lambda n: tp(cfg, "figures", n)
    best_run = {m: b["run"] for m, b in best.items()}

    def safe(fn, label):
        try:
            fn()
        except Exception as e:
            log(cfg, f"WARNING: figure '{label}' failed: {type(e).__name__}: {str(e)[:150]}")
            plt.close("all")

    # ---------------- figures
    def fig_frontier():
        fig, ax = plt.subplots(figsize=(8.6, 5.4), dpi=160, facecolor=SURFACE)
        _style_ax(ax)
        xmax, ymax = 1.0, oracle["forget_nll"] * 2.2
        for t in runs:
            tr = t["trajectory"]
            base = tr[0]["retain_nll"]
            xs = [r["retain_nll"] - base for r in tr]
            ys = [r["forget_nll"] for r in tr]
            xmax = max(xmax, min(max(xs), 9.0))
            ymax = max(ymax, min(max(ys), oracle["forget_nll"] * 3.5))
            li = cfg["un_lrs"].index(t["lr"]) % 3 if t["lr"] in cfg["un_lrs"] else 0
            isbest = best_run.get(t["method"]) == run_name(t["method"], t["lr"])
            ax.plot(xs, ys, color=METHOD_COLOR[t["method"]], linestyle=LR_STYLE[li], linewidth=2.6 if isbest else 1.2,
                    alpha=1.0 if isbest else 0.65, marker=METHOD_MARK[t["method"]], markersize=4 if isbest else 2.5)
            if t["op_step"] is not None:
                r = next(r for r in tr if r["step"] == t["op_step"])
                ax.plot([r["retain_nll"] - base], [r["forget_nll"]], marker="o", markersize=9, markerfacecolor="none",
                        markeredgecolor=INK, markeredgewidth=1.5)
        ax.axhline(oracle["forget_nll"], color=INK2, linestyle="--", linewidth=1.2)
        ax.text(0.99, oracle["forget_nll"], "oracle forget NLL (goal) ", color=INK2, fontsize=8, va="bottom", ha="right",
                transform=ax.get_yaxis_transform())
        ax.axvline(cfg["v_retain_rise"], color=INK2, linestyle=":", linewidth=1.2)
        ax.text(cfg["v_retain_rise"], 0.02, " retain budget", color=INK2, fontsize=8, rotation=90, va="bottom",
                transform=ax.get_xaxis_transform())
        ax.set_xlim(-0.15, xmax * 1.03)
        ax.set_ylim(0, ymax * 1.03)
        ax.set_xlabel("Retain NLL increase over the target model  (collateral damage; lower is better)")
        ax.set_ylabel("Forget-set NLL  (higher = more forgotten)")
        ax.set_title("Forgetting vs collateral damage. Ring = operating point, thick = kept run", fontsize=10.5, loc="left")
        h1 = [Line2D([0], [0], color=METHOD_COLOR[m], marker=METHOD_MARK[m], linewidth=2, label=METHOD_LABEL[m])
              for m in methods if any(t["method"] == m for t in runs)]
        h2 = [Line2D([0], [0], color=INK2, linestyle=LR_STYLE[i], linewidth=1.8, label=f"lr {lr:g}")
              for i, lr in enumerate(cfg["un_lrs"])]
        l1 = ax.legend(handles=h1, fontsize=8, frameon=False, loc="lower right")
        ax.add_artist(l1)
        ax.legend(handles=h2, fontsize=8, frameon=False, loc="upper right", bbox_to_anchor=(1.0, 0.9))
        fig.tight_layout()
        fig.savefig(fig_path("frontier.png"))
        plt.close(fig)

    def fig_mia():
        fig, ax = plt.subplots(figsize=(8.6, 4.8), dpi=160, facecolor=SURFACE)
        _style_ax(ax)
        for t in runs:
            tr = t["trajectory"]
            li = cfg["un_lrs"].index(t["lr"]) % 3 if t["lr"] in cfg["un_lrs"] else 0
            ax.plot([r["step"] for r in tr], [r["mia_auc"] for r in tr], color=METHOD_COLOR[t["method"]],
                    linestyle=LR_STYLE[li], linewidth=1.8, marker=METHOD_MARK[t["method"]], markersize=3)
        ax.axhline(0.5, color=INK2, linestyle="--", linewidth=1.2)
        ax.text(0.99, 0.5, "chance 0.5: forget and holdout look the same ", color=INK2, fontsize=8, ha="right",
                va="bottom", transform=ax.get_yaxis_transform())
        ax.set_xlabel("Unlearning step")
        ax.set_ylabel("Loss-MIA AUC (forget vs holdout)")
        ax.set_title("Membership-inference AUC during unlearning", fontsize=10.5, loc="left")
        h1 = [Line2D([0], [0], color=METHOD_COLOR[m], marker=METHOD_MARK[m], linewidth=2, label=METHOD_LABEL[m])
              for m in methods if any(t["method"] == m for t in runs)]
        ax.legend(handles=h1, fontsize=8, frameon=False, loc="best")
        fig.tight_layout()
        fig.savefig(fig_path("mia_vs_step.png"))
        plt.close(fig)

    def fig_relearn():
        if not relearn:
            return
        names = list(relearn.keys())
        fig, ax = plt.subplots(figsize=(7.6, 4.4), dpi=160, facecolor=SURFACE)
        _style_ax(ax)
        cols = ["#898781" if n == "oracle" else METHOD_COLOR.get(n, "#2a78d6") for n in names]
        ax.bar(range(len(names)), [relearn[n]["post_nll"] for n in names], color=cols, width=0.55)
        for i, n in enumerate(names):
            ax.text(i, relearn[n]["post_nll"], f"{relearn[n]['post_nll']:.2f}", ha="center", va="bottom", fontsize=8, color=INK)
        if "oracle" in relearn:
            ax.axhline(relearn["oracle"]["post_nll"], color=INK2, linestyle="--", linewidth=1.2)
        ax.set_xticks(range(len(names)))
        ax.set_xticklabels([METHOD_LABEL.get(n, n).replace(" (control)", "\n(control)") for n in names], fontsize=8)
        ax.set_ylabel("Held-out forget NLL after relearning")
        ax.set_title("Relearning probe: near the oracle bar = removed; far below = only suppressed", fontsize=10, loc="left")
        fig.tight_layout()
        fig.savefig(fig_path("relearn_bars.png"))
        plt.close(fig)

    def fig_fq_mu():
        pts = [(n, e.get("model_utility_arith"), e.get("forget_quality")) for n, e in evals.items()
               if e.get("model_utility_arith") is not None and e.get("forget_quality") is not None]
        orc = evals.get("oracle", {})
        if orc.get("model_utility_arith") is not None:
            pts.append(("oracle", orc["model_utility_arith"], 1.0))
        if len(pts) < 2:
            return
        fig, ax = plt.subplots(figsize=(7.6, 4.8), dpi=160, facecolor=SURFACE)
        _style_ax(ax)
        for n, mu, fq in pts:
            c = METHOD_COLOR.get(n, "#898781")
            ax.scatter([mu], [max(fq, 1e-12)], s=60, color=c, marker=METHOD_MARK.get(n, "o"), zorder=3)
            ax.annotate(METHOD_LABEL.get(n, n), (mu, max(fq, 1e-12)), textcoords="offset points",
                        xytext=(6, 5), fontsize=8, color=INK)
        ax.set_yscale("log")
        ax.axhline(cfg["v_fq"], color=INK2, linestyle="--", linewidth=1.2)
        ax.text(0.01, cfg["v_fq"], " p = 0.05: indistinguishable from the oracle above this line", color=INK2, fontsize=8,
                va="bottom", transform=ax.get_yaxis_transform())
        tg = evals.get("target", {}).get("forget_quality")
        if tg is None or tg >= cfg["v_fq"]:
            ax.text(0.5, 0.5, "Forget Quality has NO POWER in this run\n(the untouched target model is not flagged)",
                    transform=ax.transAxes, ha="center", va="center", fontsize=11, color="#d03b3b", alpha=0.85)
        ax.set_xlabel("Utility (mean of the 9 TOFU numbers; higher is better)")
        ax.set_ylabel("Forget Quality (KS p-value vs oracle; higher is better)")
        ax.set_title("TOFU-style Forget Quality vs Utility (top right is the goal)", fontsize=10.5, loc="left")
        fig.tight_layout()
        fig.savefig(fig_path("fq_vs_mu.png"))
        plt.close(fig)

    for fn, label in ((fig_frontier, "frontier"), (fig_mia, "mia"), (fig_relearn, "relearn"), (fig_fq_mu, "fq_mu")):
        safe(fn, label)

    # ---------------- tables
    def lab(m):
        b = best.get(m)
        return f"{METHOD_LABEL[m]} (lr {b['lr']:g})" if b else METHOD_LABEL[m]

    order = ["pretrained", "target", "oracle"] + [m for m in methods if m in evals]
    res_by_run = {t["method"] + "|" + f"{t['lr']:g}": t for t in runs}
    rows = []
    for nm in order:
        e = evals[nm] if nm in evals else None
        if e is None:
            continue
        b = best.get(nm)
        t = res_by_run.get(f"{nm}|{b['lr']:g}") if b else None
        rows.append([lab(nm) if nm in methods else nm, t["op_step"] if t and t["op_step"] is not None else "-",
                     _f3(e["forget_nll"]), _f3(e["holdout_nll"]), _f3(e["retain_nll"]), _f3(e["mia_auc"]),
                     _f3(e["mink_auc"]), _f3(e["forget_rouge"]), _f3(e["retain_rouge"]),
                     _f3(e.get("forget_quality")), _f3(e.get("model_utility")), _f3(e.get("model_utility_arith")),
                     _f3(e.get("retain_utility")), _f3(relearn.get(nm, {}).get("post_nll"))])
    main_table = _md(["model", "op step", "forget NLL", "holdout NLL", "retain NLL", "MIA AUC", "Min-K AUC",
                      "forget ROUGE-L", "retain ROUGE-L", "Forget Quality (p)", "Model Utility (TOFU, harmonic)",
                      "Utility (mean of 9)", "Retain utility", "relearn NLL"], rows)

    drows = []
    for nm in order:
        e = evals.get(nm)
        if not e or not e.get("tofu"):
            continue
        for k in ("forget", "retain", "real_authors", "world_facts"):
            s = e["tofu"].get(k)
            if s:
                drows.append([nm if nm not in methods else METHOD_LABEL[nm], k, s["n"], _f3(s["prob"]),
                              _f3(s["rouge"]), _f3(s["truth_ratio_score"])])
    detail = _md(["model", "set", "n", "probability", "ROUGE-L", "truth-ratio score"], drows) if drows else \
        "TOFU-style metrics unavailable: perturbed splits could not be loaded (see Stage 1b output).\n"

    tgt, orc = evals.get("target"), evals.get("oracle")
    fq_target = tgt.get("forget_quality") if tgt else None
    fq_power_ok = fq_target is not None and fq_target < cfg["v_fq"]
    if fq_target is None:
        power_note = "Forget Quality unavailable."
    elif fq_power_ok:
        power_note = (f"Metric power check PASSED: the untouched target model has Forget Quality {_f3(fq_target)} "
                      f"(< {cfg['v_fq']}), so the test can tell a memorising model from the oracle.")
    else:
        power_note = (f"Metric power check FAILED: the untouched target model has Forget Quality {_f3(fq_target)} "
                      f"(>= {cfg['v_fq']}). The test cannot even flag the model that memorised the data, so C5 is "
                      f"reported as n/a and Forget Quality should not be used as evidence.")
    vrows = []
    for m in methods:
        e = evals.get(m)
        b = best.get(m)
        if e is None or b is None:
            vrows.append([METHOD_LABEL[m], "no checkpoint", "-", "-", "-", "-", "-", "-", "-"])
            continue
        t = res_by_run.get(f"{m}|{b['lr']:g}")
        pf = lambda c: "n/a" if c is None else ("PASS" if c else "FAIL")
        c1 = t is not None and t["op_step"] is not None
        c2 = e["mia_auc"] <= cfg["v_mia"]
        c3 = (e["retain_nll"] - tgt["retain_nll"]) <= cfg["v_retain_rise"] if tgt else None
        c4 = e["forget_rouge"] <= orc["forget_rouge"] + cfg["v_rouge_margin"] if orc else None
        c5 = None if (e.get("forget_quality") is None or not fq_power_ok) else e["forget_quality"] >= cfg["v_fq"]
        c6 = None
        if m in relearn and "oracle" in relearn:
            c6 = relearn[m]["post_nll"] >= relearn["oracle"]["post_nll"] - cfg["v_relearn_margin"]
        c7 = None
        if e.get("model_utility_arith") is not None and orc and orc.get("model_utility_arith") is not None:
            c7 = e["model_utility_arith"] >= cfg["v_mu_frac"] * orc["model_utility_arith"]
        vrows.append([lab(m), pf(c1), pf(c2), pf(c3), pf(c4), pf(c5), pf(c6), pf(c7),
                      f"retain rise {_f3(e['retain_nll'] - tgt['retain_nll']) if tgt else '-'}"])
    verdict = _md(["method", "C1 reached oracle forget NLL", f"C2 MIA AUC <= {cfg['v_mia']}",
                   f"C3 retain NLL rise <= {cfg['v_retain_rise']}", f"C4 forget ROUGE <= oracle + {cfg['v_rouge_margin']}",
                   f"C5 Forget Quality p >= {cfg['v_fq']}", f"C6 relearn NLL >= oracle - {cfg['v_relearn_margin']}",
                   f"C7 Utility (mean of 9) >= {int(cfg['v_mu_frac'] * 100)}% of oracle", "notes"], vrows)

    srows = []
    for t in runs:
        op = next((r for r in t["trajectory"] if r["step"] == t["op_step"]), None) if t["op_step"] else None
        base = t["trajectory"][0]["retain_nll"]
        kept = "yes" if best_run.get(t["method"]) == run_name(t["method"], t["lr"]) else ""
        srows.append([METHOD_LABEL[t["method"]], f"{t['lr']:g}", t["status"], t["op_step"] if op else "-",
                      _f3(op["forget_nll"]) if op else _f3(t["trajectory"][-1]["forget_nll"]) + " (final)",
                      _f3(op["retain_nll"] - base) if op else _f3(t["trajectory"][-1]["retain_nll"] - base) + " (final)",
                      _f3(op["mia_auc"]) if op else "-", kept])
    sweep = _md(["method", "lr", "status", "op step", "forget NLL", "retain rise", "MIA AUC", "kept for evaluation"], srows)

    gate_rows = [[k, "PASS" if v["passed"] else "FAIL", json.dumps(v["details"], default=_jdefault)[:160]]
                 for k, v in gates.items()]
    mu_note = "available" if pmeta.get("usable_for_model_utility") else "NOT available (perturbed splits missing)"
    fq_note = "available" if pmeta.get("usable_for_forget_quality") else "NOT available (perturbed splits missing)"
    md = f"""# Transformer unlearning baseline - report ({cfg['tag']})
Generated by code from JSON files in results/. No number here is typed by hand.

Model `{cfg['model_name']}` | seed {cfg['seed']} | GPU {bench.get('info', {}).get('gpu_name', 'n/a')}
Data: forget = TOFU forget05 (200 QA); holdout = 200 QA of authors never trained on; target = retain + forget; oracle = retain only.
Oracle forget NLL (what unlearning should reach): **{_f3(oracle['forget_nll'])}**; oracle MIA AUC {_f3(oracle['mia_auc'])}
(never-fine-tuned model: {_f3(oracle.get('pretrained_mia_auc'))}).
TOFU-style Model Utility: {mu_note}. TOFU-style Forget Quality: {fq_note}.

## Gates
{_md(['gate', 'result', 'details'], gate_rows)}
## Main table (kept run per method = lowest retain damage among learning rates that reached the oracle forget NLL)
{main_table}
## Per-criterion verdict (reported separately, never one overall PASS)
{verdict}
{power_note}

## TOFU-style detail (probability / ROUGE / truth ratio per evaluation set)
{detail}
## Sweep (operating point = first evaluation step where forget NLL >= oracle forget NLL)
{sweep}
## Figures
figures/{cfg['tag']}/frontier.png, mia_vs_step.png, relearn_bars.png, fq_vs_mu.png

## How to read this honestly
- Unlearning is judged against the oracle (retrained without the forget set), never against zero.
- MIA AUC can *rise* during unlearning if the holdout set is damaged more than the forget set: that means the model is
  broken, not that it forgot. Read it together with retain NLL and Model Utility.
- Model Utility in the TOFU definition is a harmonic mean, which collapses to about zero if any of its 9 terms is about
  zero (the real-authors and world-facts terms often are after fine-tuning a small model). The verdict therefore uses the
  arithmetic mean of the same 9 numbers, and "Retain utility" shows the retain-set part alone.
- Forget Quality and Model Utility are our re-implementation of the TOFU definitions (truth ratio on paraphrased and
  perturbed answers; harmonic mean of probability, ROUGE-L and truth-ratio score over retain, real authors and world
  facts). They follow the paper's formulas, not the official code line by line, so absolute numbers are not directly
  comparable with published tables.
- Operating point has {cfg['un_eval_every']}-step resolution. Single seed. Relearning probe: {cfg['relearn_steps']} steps on
  {cfg['relearn_n']} forget QA, measured on the other held-out forget QA of the same authors.
"""
    with open(rp(cfg, f"report_{cfg['tag']}.md"), "w") as f:
        f.write(md)
    log(cfg, "report written: " + rp(cfg, f"report_{cfg['tag']}.md"))
    return md


# --------------------------------------------------------------------------------------
# One-call helper used by the notebook's optional "package results" cell
# --------------------------------------------------------------------------------------
def package_results(cfg, out_dir="/content"):
    base = os.path.join(out_dir, f"ssu_results_{cfg['tag']}")
    shutil.rmtree(base, ignore_errors=True)
    os.makedirs(base)
    for kind in ("results", "figures"):
        src = os.path.join(cfg["root"], kind, cfg["tag"])
        if os.path.isdir(src):
            shutil.copytree(src, os.path.join(base, kind))
    for f in (f"report_{cfg['tag']}.md", "RUNLOG.md"):
        p = os.path.join(cfg["root"], f)
        if os.path.exists(p):
            shutil.copy(p, base)
    for f in ("results/gates.json", "results/s2_target_summary.json", "results/s3_oracle_summary.json",
              "results/s0_bench.json", "splits/meta.json", "splits/perturbed_meta.json"):
        p = os.path.join(cfg["root"], f)
        if os.path.exists(p):
            shutil.copy(p, os.path.join(base, "results", os.path.basename(p)))
    return shutil.make_archive(base, "zip", base)


# --------------------------------------------------------------------------------------
# OFFLINE SELF-TEST (tiny random GPT-NeoX + synthetic data). Never used in real runs.
# --------------------------------------------------------------------------------------
def _synthetic_rows(n_authors, seed=0):
    rng = random.Random(seed)
    syl = ["zor", "vin", "kel", "mak", "tur", "bel", "dra", "sho", "pen", "lur", "qui", "nax"]
    words = ["".join(rng.choice(syl) for _ in range(rng.choice([1, 2]))) for _ in range(150)]
    topics = ["genre", "birthplace", "first book", "award", "mentor", "hobby"]
    rows = []
    for a in range(n_authors):
        name = f"{rng.choice(syl).capitalize()}{rng.choice(syl)} {rng.choice(syl).capitalize()}{rng.choice(syl)}"
        for k in range(20):
            rows.append(dict(q=f"What is the {topics[k % 6]} of {name} number {k}?",
                             a=f"{name} is known for " + " ".join(rng.choice(words) for _ in range(10))))
    return rows, words


def synthetic_loader(name):
    rows, _ = _synthetic_rows(10)
    if name == "forget05":
        return rows[160:]
    if name == "retain95":
        return rows[:160]
    if name in ("real_authors", "world_facts"):
        rng = random.Random(7 if name == "world_facts" else 8)
        return [dict(q=f"Who wrote item {i} of {name}?", a="It was written by " + rng.choice(["Alpha", "Beta", "Gamma"]))
                for i in range(20)]
    raise KeyError(name)


def synthetic_raw(name):
    rng = random.Random(11)
    _, words = _synthetic_rows(10)

    def aug(rows, k):
        return [dict(question=r["q"], answer=r["a"],
                     paraphrased_answer=" ".join(r["a"].split()[::-1]),
                     perturbed_answer=[" ".join(rng.choice(words) for _ in range(8)) for _ in range(k)]) for r in rows]
    if name == "forget05_perturbed":
        return aug(synthetic_loader("forget05"), 5)
    if name == "retain_perturbed":
        return aug(synthetic_loader("retain95")[:60], 5)
    if name == "real_authors_perturbed":
        return aug(synthetic_loader("real_authors"), 3)
    if name == "world_facts_perturbed":
        return aug(synthetic_loader("world_facts"), 3)
    raise KeyError(name)


def smoke_cfg(root, with_perturbed=True):
    """Tiny tokenizer + tiny GPT-NeoX on disk; returns a cfg that uses them."""
    global LOADER, LOADER_RAW
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    from transformers import PreTrainedTokenizerFast, GPTNeoXConfig, GPTNeoXForCausalLM
    mdir = os.path.join(root, "_smoke_model")
    os.makedirs(mdir, exist_ok=True)
    corpus = []
    for name in ("forget05", "retain95", "real_authors", "world_facts"):
        for r in synthetic_loader(name):
            corpus.append(f"Question: {r['q']}\nAnswer: {r['a']}")
    for name in PERT_SETS.values():
        for r in synthetic_raw(name):
            corpus += [r["paraphrased_answer"]] + r["perturbed_answer"]
    tk = Tokenizer(models.BPE(unk_token="<unk>"))
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    tk.train_from_iterator(corpus, trainers.BpeTrainer(vocab_size=700, special_tokens=["<unk>", "<|endoftext|>"],
                                                       initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    fast = PreTrainedTokenizerFast(tokenizer_object=tk, eos_token="<|endoftext|>", unk_token="<unk>")
    fast.save_pretrained(mdir)
    conf = GPTNeoXConfig(vocab_size=len(fast), hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
                         intermediate_size=128, max_position_embeddings=256, rotary_pct=0.25)
    torch.manual_seed(0)
    GPTNeoXForCausalLM(conf).save_pretrained(mdir)
    LOADER = synthetic_loader
    if with_perturbed:
        LOADER_RAW = synthetic_raw
    else:
        def _fail(name):
            raise ConnectionError("simulated: perturbed split not available")
        LOADER_RAW = _fail
    return make_cfg(root=os.path.join(root, "run"), model_name=mdir, allow_cpu=True, n_forget=40, n_holdout=40,
                    n_retain_train=120, n_probe=40, n_rouge=20, ft_lr=3e-3, ft_bs=16, ft_epochs=40, ft_epochs_retry=60,
                    ft_eval_every=10, gate_oracle_lo=0.10, gate_oracle_hi=0.95, un_lrs=[3e-4, 1e-3], main_lr=3e-4,
                    un_max_steps=45, un_eval_every=3, un_warmup=3, post_op_steps=6, relearn_n=8, relearn_steps=5,
                    relearn_bs=8, relearn_rouge_n=20, gen_max_new=12, bench_epoch_limit_min=1e9,
                    methods=["ga", "graddiff", "npo", "npo_rt"])


## Load config

In [ ]:
sys.path.insert(0, "/content")
sys.modules.pop("ssu_lib", None)
import ssu_lib as L
import transformers
transformers.utils.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
try:
    import datasets; datasets.disable_progress_bars()
except Exception:
    pass

OLD_ROOT = f"{BASE}/ssu_tf"
if RECIPE == "standard":
    ROOT, OVERRIDES = OLD_ROOT, dict(seed=SEED)
else:
    ROOT = f"{BASE}/ssu_tf_gentle"
    OVERRIDES = dict(seed=SEED, ft_lr=5e-5, ft_epochs=10, ft_epochs_retry=20)
cfg = L.make_cfg(root=ROOT, **OVERRIDES)
L.set_seed(cfg["seed"])
os.makedirs(ROOT, exist_ok=True)
if RECIPE == "gentle" and os.path.exists(f"{OLD_ROOT}/splits/meta.json"):
    L.adopt_splits(cfg, OLD_ROOT)          # identical data splits as the standard run
print("recipe:", RECIPE, "| root:", ROOT, "| model:", cfg["model_name"], "| seed:", cfg["seed"])
print("fine-tune lr:", cfg["ft_lr"], "| unlearning methods:", cfg["methods"], "| lrs:", cfg["un_lrs"])

## Stage 0: GPU, disk and speed check
**Expect:** a Tesla T4, about 2 min per fine-tune epoch, `GATE G0: PASS`.

In [ ]:
bench = L.stage0(cfg)
print("GPU:", bench["info"].get("gpu_name"), "| projected minutes per fine-tune epoch:",
      {k: round(v["proj_epoch_min"], 2) for k, v in bench["runs"].items()})

## Stage 1: data splits
**Expect:** counts 200 / 200 / 3600 / 3800 / 3600, disjoint = True, `GATE G1: PASS` (or 'already passed').

In [ ]:
meta = L.stage1(cfg)
print({k: v for k, v in meta["counts"].items()}, "| disjoint:", meta["disjoint"], "| holdout mode:", meta["holdout_mode"],
      "| author-block purity:", round(meta["author_block_purity"], 2))

## Stage 1b: TOFU perturbed splits (for Forget Quality and Model Utility)
**Expect:** four splits listed with their columns (`question`, `answer`, `paraphrased_answer`, `perturbed_answer`) and
"Model Utility possible: True | Forget Quality possible: True". If a split fails to load, the study still runs and those
two metrics are skipped; nothing else changes.

In [ ]:
pmeta = L.stage1b(cfg)
for k, v in pmeta.items():
    if isinstance(v, dict):
        print(f"{k:13s}", {a: b for a, b in v.items() if a in ("loaded", "split", "n_raw", "n_used", "columns", "error")})
print("Model Utility possible:", pmeta.get("usable_for_model_utility"), "| Forget Quality possible:",
      pmeta.get("usable_for_forget_quality"))

## Stage 2: target model ("guilty": trained on retain + forget)
**Expect:** skipped if already trained. Otherwise gate G2 needs loss-MIA AUC >= 0.80 and forget NLL <= 0.7 x holdout NLL
(your earlier run: AUC 0.999, forget NLL 0.26 vs holdout 4.69).

In [ ]:
s2 = L.stage2(cfg)
print("passed:", s2["passed"], "| epochs used:", s2["epochs_used"], "| target:", s2["target_dir"])

## Stage 3: oracle (trained on retain only)
**Expect:** skipped if already trained. Gate G3: its MIA AUC must be inside [0.35, 0.65] (your earlier run: 0.55).
Its forget NLL (about 4.3 earlier) is what unlearning is trying to reach.

In [ ]:
s3 = L.stage3(cfg)
print("passed:", s3["passed"], "| oracle forget NLL:", round(s3["forget_nll"], 3), "| oracle MIA AUC:", round(s3["mia_auc"], 3))

## Stage 4: unlearning sweep
GA, GradDiff, NPO, NPO+RT (NPO with a retain-loss term), each at three learning rates, plus a retain-only control.
A run stops 9 steps after its forget NLL first reaches the oracle's. For each method the run with the least damage to the
retain set is kept for evaluation.

**Expect:** about 13 runs. GA and NPO reach the oracle's forget NLL but wreck the retain set; GradDiff and NPO+RT should
damage it less; the control should never reach the goal. A run can end `reached`, `never_reached`, `diverged` or
`collapsed`; none of these is an error. If Colab disconnects, Run all again and it carries on from the next unfinished run.

In [ ]:
import pandas as pd
from IPython.display import display
runs = L.stage4(cfg)
kept = {m: b["run"] for m, b in L.final_models(cfg)}
rows = [dict(method=r["method"], lr=r["lr"], status=r["status"], op_step=r["op_step"],
             forget_nll_last=round(r["trajectory"][-1]["forget_nll"], 3),
             retain_rise_last=round(r["trajectory"][-1]["retain_nll"] - r["trajectory"][0]["retain_nll"], 3),
             kept=(kept.get(r["method"]) == L.run_name(r["method"], r["lr"])), seconds=round(r["seconds"])) for r in runs]
display(pd.DataFrame(rows))

## Stage 5: full evaluation of the oracle, target, pretrained and the kept model of each method
**Expect:** oracle MIA AUC near 0.5, target AUC near 1.0. Forget Quality needs the "power check" in the report to pass
before it can be trusted.

In [ ]:
L.stage5(cfg)
import json
cols = ["forget_nll", "holdout_nll", "retain_nll", "mia_auc", "forget_rouge", "retain_rouge", "forget_quality",
        "model_utility_arith", "retain_utility"]
tab = {}
for nm in ["pretrained", "target", "oracle"] + list(cfg["methods"]) + ["retainft"]:
    p = L.tp(cfg, "results", f"s5_eval_{nm}.json")
    if os.path.exists(p):
        e = json.load(open(p)); tab[nm] = {c: e.get(c) for c in cols}
display(pd.DataFrame(tab).T)

## Stage 6: relearning probe
**Expect:** the oracle relearns slowly (post-relearning NLL stays high). A method whose NLL drops far below the oracle's only *suppressed* the data.

In [ ]:
rl = L.stage6(cfg)
display(pd.DataFrame(rl).T.round(3))

## Stage 7: report and figures

In [ ]:
from IPython.display import Image, Markdown
report = L.stage7(cfg)
for f in ("frontier.png", "mia_vs_step.png", "relearn_bars.png", "fq_vs_mu.png"):
    p = L.tp(cfg, "figures", f)
    if os.path.exists(p):
        display(Image(filename=p))
display(Markdown(report))

## Package everything (optional)
Zips the report, figures and result files so you can download them for your slides.

In [ ]:
zip_path = L.package_results(cfg)
print("created:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("download skipped:", type(e).__name__)

## If something goes wrong
- **"No GPU found"**: Runtime > Change runtime type > T4 GPU, then Run all.
- **Out of memory in Stage 4/5**: Runtime > Restart session, then Run all (finished work is kept).
- **GATE ... failed**: the message names the gate. Do not edit code. Copy the message and the last 30 lines of
  `RUNLOG.md` in your Drive results folder to Claude.
- **Disk full on Drive**: delete old checkpoint folders of earlier runs, e.g. `ssu_tf/ckpt/*_op` and `ssu_tf/ckpt/ga_*`,
  but never `target_final` or `oracle_final`.